In [1]:
import os
import base64
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow  # pyright: ignore[reportMissingImports]
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

In [2]:
def gmail_authenticate():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

In [3]:
def get_recent_emails(service, max_results=4):
    results = service.users().messages().list(
        userId='me', maxResults=max_results, labelIds=['INBOX']
    ).execute()
    messages = results.get('messages', [])

    emails = []
    for msg in messages:
        msg_data = service.users().messages().get(
            userId='me', id=msg['id'], format='full'
        ).execute()

        headers = msg_data['payload']['headers']
        subject = next((h['value'] for h in headers if h['name'] == 'Subject'), 'بدون عنوان')
        sender = next((h['value'] for h in headers if h['name'] == 'From'), 'غير معروف')

        body = extract_body(msg_data['payload'])
        emails.append({'subject': subject, 'from': sender, 'body': body})
    return emails

In [4]:
def extract_body(payload):
    if 'parts' in payload:
        for part in payload['parts']:
            if part['mimeType'] == 'text/plain':
                data = part['body'].get('data', '')
                if data:
                    return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
    else:
        data = payload['body'].get('data', '')
        if data:
            return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
    return ''

In [6]:
import ollama

def analyze_email(email):
    prompt = f"""Analyze this email and get back to me:
1. Short summary (maximum two lines)
2. Classification: (Business / Personal / Advertisements / Urgent / Spam)
3. Importance level: (High / Medium / Low)
From: {email['from']}
Topic: {email['subject']}
Content: {email['body'][:1500]}
"""
    response = ollama.chat(
        model='gemma4:e2b',
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response['message']['content']

def main():
    service = gmail_authenticate()
    emails = get_recent_emails(service, max_results=4)

    results = []
    for email in emails:
        analysis = analyze_email(email)
        results.append({'email': email, 'analysis': analysis})

    for r in results:
        print(f"\n{'='*50}")
        print(f"From:{r['email']['from']}")
        print(f"Topic: {r['email']['subject']}")
        print(f"\n Analysis\n{r['analysis']}")

if __name__ == '__main__':
    main()


From:LinkedIn <messages-noreply@linkedin.com>
Topic: View Eman Salem’s post and your next steps

 Analysis
**1. Short summary (maximum two lines)**
This is a LinkedIn notification prompting Ahmed to view recent posts from Eman Salem and providing information about the updates and network messages available on the platform.

**2. Classification:** Business

**3. Importance level:** Low (It is an automated notification and content recommendation, not an urgent action item.)

From:Oracle <no-reply@identity.oci.oraclecloud.com>
Topic: Your Oracle One-Time Passcode

 Analysis
**1. Short summary (maximum two lines)**
This email is a notification from Oracle regarding an Oracle One-Time Passcode. It appears to be an automated system message related to Oracle Cloud identity.

**2. Classification:** Business

**3. Importance level:** High (If you are an Oracle user, this message is related to your account security and access.)

From:LinkedIn <messages-noreply@linkedin.com>
Topic: Radwa Akram, 